# Entra ID Inbound and Outbound Authentication with AgentCore

This notebook demonstrates how to configure an agent and MCP server on Amazon Bedrock AgentCore Runtime with Microsoft Entra ID for both inbound (user→agent) and outbound (agent→MCP) authentication.

## Architecture

```
┌──────────┐    ID Token     ┌─────────────────┐    M2M Token    ┌─────────────────┐
│   User   │ ───────────────►│  AgentCore      │ ───────────────►│  AgentCore      │
│ (Browser)│  (Entra ID)     │  Runtime Agent  │  (Credential    │  Runtime MCP    │
└──────────┘                 └─────────────────┘   Provider)      └─────────────────┘
```

## Learning Objectives
1. Setup Entra ID app registrations for user auth and M2M auth
2. Create an MCP server on AgentCore Runtime with M2M token validation
3. Create an agent that uses AgentCore Credential Provider for outbound M2M tokens
4. Invoke the agent with user authentication

## Prerequisites

### Entra ID Setup

You need **three** app registrations in Entra ID:

1. **User App** - For user authentication to the agent
   - Enable ID tokens and Access tokens
   - Add redirect URI for AgentCore callback
   
2. **MCP Server App** - For M2M token validation
   - Expose an API with Application ID URI: `api://<mcp-client-id>`
   - Add scope: `access_as_service`
   
3. **Agent Service App** - Service principal for M2M
   - Create client secret
   - Grant permission to MCP Server App's scope

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import os
import uuid
import boto3
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
sts = boto3.client('sts')
account_id = sts.get_caller_identity().get("Account")
region = boto_session.region_name or "us-west-2"

## Step 1: Configure Environment Variables

Set your Entra ID configuration. Replace the placeholder values with your actual Entra ID app registration details.

In [ ]:
# User App (for inbound auth)
os.environ["ENTRA_TENANT_ID"] = "your-tenant-id"  # Replace
os.environ["ENTRA_USER_APP_ID"] = "your-user-app-client-id"  # Replace
os.environ["ENTRA_USER_APP_SECRET"] = "your-user-app-secret"  # Replace

# MCP Server App (for M2M validation)
os.environ["ENTRA_MCP_APP_ID"] = "your-mcp-app-client-id"  # Replace

# Agent Service App (for M2M token acquisition)
os.environ["ENTRA_AGENT_CLIENT_ID"] = "your-agent-client-id"  # Replace
os.environ["ENTRA_AGENT_CLIENT_SECRET"] = "your-agent-client-secret"  # Replace

# Scopes
os.environ["ENTRA_USER_SCOPE"] = f"api://{os.environ['ENTRA_USER_APP_ID']}/.default openid profile"

## Step 2: Create MCP Server Code

This MCP server provides simple tools and validates M2M tokens from the agent.

In [ ]:
%%writefile mcp_server.py
"""MCP Server with Entra ID M2M Authentication"""
from mcp.server.fastmcp import FastMCP
from typing import Dict, Any

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def get_greeting(name: str) -> Dict[str, Any]:
    """Get a personalized greeting."""
    return {"greeting": f"Hello, {name}! Welcome to the Entra ID authenticated MCP server."}

@mcp.tool()
def add_numbers(a: int, b: int) -> Dict[str, Any]:
    """Add two numbers together."""
    return {"result": a + b}

@mcp.tool()
def get_server_info() -> Dict[str, Any]:
    """Get information about this MCP server."""
    return {
        "server": "Entra ID Sample MCP Server",
        "auth_type": "Entra ID M2M (Client Credentials)",
        "tools": ["get_greeting", "add_numbers", "get_server_info"]
    }

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

## Step 3: Deploy MCP Server to AgentCore Runtime

Configure the MCP server with Entra ID M2M token validation.

In [ ]:
mcp_runtime = Runtime()

# Entra ID discovery URL for M2M tokens (v1.0 endpoint)
mcp_discovery_url = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/.well-known/openid-configuration"
mcp_audience = f"api://{os.environ['ENTRA_MCP_APP_ID']}"

mcp_config = mcp_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="entra_id_sample_mcp",
    protocol_configuration={"serverProtocol": "MCP"},
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": mcp_discovery_url,
            "allowedAudience": [mcp_audience]
        }
    }
)
print(f"MCP Runtime configured")

In [ ]:
# Deploy MCP Server
mcp_launch = mcp_runtime.launch(local_build=True)
print(f"MCP Server deployed: {mcp_launch.agent_id}")

## Step 4: Create Credential Provider for M2M

Create an AgentCore credential provider that the agent will use to obtain M2M tokens.

In [ ]:
import urllib.parse

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

# Create credential provider for M2M
try:
    cp_response = agentcore_client.create_oauth2_credential_provider(
        name="entra-id-m2m-provider",
        credentialProviderVendor="CustomOAuth2",
        oauth2ProviderConfigInput={
            "customOAuth2ProviderConfig": {
                "tokenUrl": f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/oauth2/v2.0/token",
                "clientId": os.environ["ENTRA_AGENT_CLIENT_ID"],
                "clientSecret": os.environ["ENTRA_AGENT_CLIENT_SECRET"]
            }
        }
    )
    print(f"Created credential provider: {cp_response['credentialProviderArn']}")
except agentcore_client.exceptions.ConflictException:
    print("Credential provider already exists")

## Step 5: Create Agent Code

The agent uses the credential provider to get M2M tokens for calling the MCP server.

In [ ]:
# Get MCP URL for agent
escaped_mcp_arn = urllib.parse.quote(mcp_launch.agent_arn, safe='')
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_mcp_arn}/invocations?qualifier=DEFAULT"
print(f"MCP URL: {mcp_url}")

In [ ]:
%%writefile agent.py
"""Agent with Entra ID Inbound/Outbound Authentication"""
import os
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

MCP_URL = os.environ.get("MCP_URL")
MCP_APP_ID = os.environ.get("ENTRA_MCP_APP_ID")

bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    temperature=0.1,
)

def get_m2m_token(workload_token: str) -> str:
    """Get M2M token using AgentCore credential provider."""
    client = boto3.client("bedrock-agentcore")
    scope = f"api://{MCP_APP_ID}/.default"
    response = client.get_resource_oauth2_token(
        workloadIdentityToken=workload_token,
        resourceCredentialProviderName="entra-id-m2m-provider",
        scopes=[scope],
        oauth2Flow="M2M",
    )
    return response["accessToken"]

@app.entrypoint
def agent_handler(payload, context):
    prompt = payload.get("prompt", "hello")
    workload_token = context.get("workload_access_token")
    
    # Get M2M token for MCP
    m2m_token = get_m2m_token(workload_token)
    
    # Create MCP client with M2M auth
    headers = {"authorization": f"Bearer {m2m_token}"}
    mcp_client = MCPClient(lambda: streamablehttp_client(MCP_URL, headers))
    
    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(model=bedrock_model, tools=tools)
        response = agent(prompt)
    
    return str(response)

if __name__ == "__main__":
    app.run()

## Step 6: Deploy Agent to AgentCore Runtime

In [ ]:
agent_runtime = Runtime()

# Entra ID discovery URL for user tokens (v2.0 endpoint)
agent_discovery_url = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/v2.0/.well-known/openid-configuration"

agent_config = agent_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="entra_id_sample_agent",
    environment_variables={
        "MCP_URL": mcp_url,
        "ENTRA_MCP_APP_ID": os.environ["ENTRA_MCP_APP_ID"]
    },
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": agent_discovery_url,
            "allowedAudience": [os.environ["ENTRA_USER_APP_ID"]]
        }
    }
)
print("Agent configured")

In [ ]:
# Deploy Agent
agent_launch = agent_runtime.launch(local_build=True)
print(f"Agent deployed: {agent_launch.agent_id}")

## Step 7: Get User Token via MSAL Device Code Flow

In [ ]:
import msal
import webbrowser

authority = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}"
scopes = [os.environ["ENTRA_USER_SCOPE"]]

msal_app = msal.PublicClientApplication(
    client_id=os.environ["ENTRA_USER_APP_ID"],
    authority=authority,
)

# Try silent auth first
result = msal_app.acquire_token_silent(scopes, account=None)

if not result:
    # Initiate device code flow
    flow = msal_app.initiate_device_flow(scopes=scopes)
    print(flow["message"])
    webbrowser.open(flow["verification_uri"])
    result = msal_app.acquire_token_by_device_flow(flow)

if "access_token" in result:
    bearer_token = result["access_token"]
    print(f"Bearer Token Received: {bearer_token[:30]}...")
else:
    print(f"Error: {result.get('error_description')}")

## Step 8: Invoke Agent with User Token

In [ ]:
import requests
import json

escaped_agent_arn = urllib.parse.quote(agent_launch.agent_arn, safe='')
agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

session_id = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {bearer_token}",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
}

response = requests.post(
    agent_url,
    data=json.dumps({"prompt": "What tools do you have available? Please list them."}),
    headers=headers
)
print(response.text)

In [ ]:
# Test a tool call
response = requests.post(
    agent_url,
    data=json.dumps({"prompt": "Please greet John and add 5 + 3"}),
    headers=headers
)
print(response.text)

## Cleanup

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)

# Delete Agent
agentcore_control.delete_agent_runtime(agentRuntimeId=agent_launch.agent_id)
print(f"Deleted agent: {agent_launch.agent_id}")

# Delete MCP Server
agentcore_control.delete_agent_runtime(agentRuntimeId=mcp_launch.agent_id)
print(f"Deleted MCP server: {mcp_launch.agent_id}")

# Delete Credential Provider
agentcore_client.delete_oauth2_credential_provider(name="entra-id-m2m-provider")
print("Deleted credential provider")

## Conclusion

In this notebook we learned how to:
- Configure Entra ID app registrations for user and M2M authentication
- Deploy an MCP server with M2M token validation
- Create an AgentCore credential provider for M2M token acquisition
- Deploy an agent that validates user tokens and obtains M2M tokens for MCP calls
- Invoke the agent using MSAL device code flow